In [ ]:
from dash_core_components import *
from dash_html_components import *
from dash import Dash
import dash.dependencies as dp
import plotly.express as ex
import numpy as np


def rank(a, k=0.4):
    a = a + 1e-9
    a *= k / a.sum(0)
    a += (1 - k) / len(a)
    a, b = np.linalg.eig(a)
    return np.abs(b[:, np.isclose(a, 1).nonzero()[0][0]].real)

    
K = []
V = np.empty((0, 0), int)
app = Dash()
app.layout = Div(
    [
        Graph("graph"),
        Div(repr(V), 'text'),
        Input("i"),
        Button("赞", "button"),
        Input("j"),
    ]
)


@app.callback(
    dp.Output("graph", "figure"),
    dp.Input("button", "n_clicks"),
    dp.State("i", "value"),
    dp.State("j", "value"),
)
def _(_, i, j):
    global V
    if i and j and i != j:
        if i not in K:
            K.append(i)
            V = np.pad(V, (0, 1))
        if j not in K:
            K.append(j)
            V = np.pad(V, (0, 1))
        V[K.index(j), K.index(i)] += 1
    return ex.bar(x=K, y=rank(V))

@app.callback(
    dp.Output("text", "children"),
    dp.Input("button", "n_clicks")
)
def _(_):
    return repr(V)


app.run_server("localhost", 8060)